In [1]:
import polars as pl

lf = pl.scan_parquet("../data/raw/yellow_tripdata_2025-01.parquet") #lazy scan - loads on collect()
df = lf.collect()

df = df.with_columns(
    ((pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime"))
        .dt.total_seconds() /60).alias("duration_min")
)

payment_type_map = {
    0:"flex_fare_trip",
    1:"credit_card",
    2:"cash",
    3:"no_charge",
    4:"dispute",
    5:"unknown",
    6:"voided_trip",
       }



df = df.with_columns(
    pl.col("payment_type")
        .replace_strict(payment_type_map, default=None)
        .alias("payment_type")


In [2]:
df.schema

Schema([('VendorID', Int32),
        ('tpep_pickup_datetime', Datetime(time_unit='us', time_zone=None)),
        ('tpep_dropoff_datetime', Datetime(time_unit='us', time_zone=None)),
        ('passenger_count', Int64),
        ('trip_distance', Float64),
        ('RatecodeID', Int64),
        ('store_and_fwd_flag', String),
        ('PULocationID', Int32),
        ('DOLocationID', Int32),
        ('payment_type', Int64),
        ('fare_amount', Float64),
        ('extra', Float64),
        ('mta_tax', Float64),
        ('tip_amount', Float64),
        ('tolls_amount', Float64),
        ('improvement_surcharge', Float64),
        ('total_amount', Float64),
        ('congestion_surcharge', Float64),
        ('Airport_fee', Float64),
        ('cbd_congestion_fee', Float64),
        ('duration_min', Float64)])

In [3]:
df.filter(pl.col("duration_min") > 0)

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,duration_min
i32,datetime[μs],datetime[μs],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,2025-01-01 00:18:38,2025-01-01 00:26:59,1,1.6,1,"""N""",229,237,1,10.0,3.5,0.5,3.0,0.0,1.0,18.0,2.5,0.0,0.0,8.35
1,2025-01-01 00:32:40,2025-01-01 00:35:13,1,0.5,1,"""N""",236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0,2.55
1,2025-01-01 00:44:04,2025-01-01 00:46:01,1,0.6,1,"""N""",141,141,1,5.1,3.5,0.5,2.0,0.0,1.0,12.1,2.5,0.0,0.0,1.95
2,2025-01-01 00:14:27,2025-01-01 00:20:01,3,0.52,1,"""N""",244,244,2,7.2,1.0,0.5,0.0,0.0,1.0,9.7,0.0,0.0,0.0,5.566667
2,2025-01-01 00:21:34,2025-01-01 00:25:06,3,0.66,1,"""N""",244,116,2,5.8,1.0,0.5,0.0,0.0,1.0,8.3,0.0,0.0,0.0,3.533333
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2,2025-01-31 23:01:48,2025-01-31 23:16:29,null,3.35,null,null,79,237,0,15.85,0.0,0.5,0.0,0.0,1.0,20.6,null,null,0.75,14.683333
2,2025-01-31 23:50:29,2025-02-01 00:17:27,null,8.73,null,null,161,116,0,28.14,0.0,0.5,0.0,0.0,1.0,32.89,null,null,0.75,26.966667
2,2025-01-31 23:26:59,2025-01-31 23:43:01,null,2.64,null,null,144,246,0,14.91,0.0,0.5,0.0,0.0,1.0,19.66,null,null,0.75,16.033333


In [4]:
df.group_by(pl.col("tpep_pickup_datetime").dt.hour().alias("hour")) \
    .agg(pl.col("duration_min").mean().alias("avg_duration")) \
    .sort("hour")

hour,avg_duration
i8,f64
0,13.785256
1,12.06919
2,12.763833
3,12.500513
4,14.155369
…,…
19,13.991177
20,13.681685
21,13.842334


In [5]:
payment_type_map = {
    0:"flex_fare_trip",
    1:"credit_card",
    2:"cash",
    3:"no_charge",
    4:"dispute",
    5:"unknown",
    6:"voided_trip",
       }

df = df.with_columns(
    pl.col("payment_type")
        .replace_strict(payment_type_map, default=None)
        .alias("payment_type")
)

df["payment_type"].value_counts()

payment_type,count
str,u32
"""flex_fare_trip""",540149
"""no_charge""",23773
"""dispute""",76481
"""unknown""",1
"""cash""",390429
"""credit_card""",2444393
